# Run control: workers, samples and full runs

One place to run the pipeline on the cluster and follow it:

1. **Settings**: what to run (a sample, a manifest such as the 100-slide slice, or every slide).
2. **Workers**: status, and update / restart when needed.
3. **Start** a run. It runs as a separate process, so closing this notebook or restarting the kernel does not stop it.
4. **Progress**: re-run this one cell whenever you want to see where the run is (done / failed / ETA / what each worker is doing).
5. **Results** once it has finished, and **Stop** if you need to.

Each slide goes through the full pipeline on a worker: label fields, SAM 3 segmentation, mask clean-up, missed-feather
check, per-feather checks, feather IDs. After the cluster part, the tables are built here, duplicates marked, scored
against the hand-checked answers where there are any, and (with `UPLOAD = True`) copied to Google Drive under
`Feather_Molt_Project/runs/<RUN_ID>/`. Inspect results in detail with `label_lab.ipynb`.

Rough timing: about 3.5 minutes per slide per worker, so with 4 workers ~1.5 h per 100 slides and ~35 h for all 2,406.
Kernel: **bird-feather-analysis (pixi)**.

## 1. Settings

In [ ]:
KIND = "manifest"                     # "sample" | "manifest" | "full"
N, SEED, MATCH = 10, 1, None          # for "sample": N random slides, or MATCH = ["A0012 1996", ...] substrings
MANIFEST = "../configs/slice_100.txt" # for "manifest": one file name per line
RUN_ID = "slice100-20261019"          # a new id starts a new run; an existing id is followed below
UPLOAD = True                         # copy the results to Google Drive at the end

## 2. Setup

In [ ]:
import sys, time
sys.path.insert(0, "..")
import pandas as pd
from IPython.display import display, Markdown
from src import cluster_control as cc
pd.set_option("display.max_colwidth", 100)
print("nodes:", cc.hosts())

## 3. Workers
`nodes`: each node over SSH (code commit on disk, worker processes, worker memory, free memory).
`workers`: Celery's view (slides running / waiting, and `new_code` = the worker has this pipeline loaded).

In [ ]:
nodes, workers = cc.worker_status()
display(nodes); display(workers)
print("waiting in the queue:", cc.queue_length())

### Update / restart (only when needed)
Set `ACTION` and run the cell; nothing happens with `ACTION = None`.
- `"update"`: fast-forward every node's repo to `BRANCH` from `SOURCE` (`"upstream"` = the nodes' origin, or a GitHub URL such as your fork). Local changes and run outputs are never touched; anything but a fast-forward is refused.
- `"restart"`: workers finish their current slide, exit and start again (this loads new code). On this cluster a watchdog also restarts stopped workers by itself within seconds.
- `"start"`: start a worker where none is running (never a second one).
Do not restart while a run is going: the slides on the workers are retried, which costs time.

In [ ]:
ACTION = None                          # None | "update" | "restart" | "start"
SOURCE, BRANCH = "upstream", "main"    # e.g. "https://github.com/christytty/bird-feather-analysis.git", "feather-judge"
NODES = None                           # None = all, or a list of node addresses from .env
if ACTION == "update":
    display(cc.update_code(SOURCE, BRANCH, NODES))
elif ACTION == "restart":
    display(cc.restart_workers(NODES))
elif ACTION == "start":
    display(cc.start_workers(NODES))
else:
    print("no action")

## 4. Start a run
Set `START = True` and run the cell once. It refuses to start a second copy of a run that is still going.

In [ ]:
START = False
names = cc.choose(KIND, N, SEED, MATCH, MANIFEST)
print(f"{KIND}: {len(names)} slides, e.g. {names[:3]}")
if START:
    ok, mine = __import__("src.prompt_lab", fromlist=["x"]).workers_have_options_support()
    if not ok:
        print("Workers are not on the new code yet (see section 3):", mine)
    else:
        rec = cc.launch_run(names, RUN_ID, upload=UPLOAD)
        print(f"started {RUN_ID}: {rec['slides']} slides, first estimate {rec['estimate_h']} h; log {rec['log']}")

## 5. Progress (re-run this cell any time)

In [ ]:
p = cc.progress(RUN_ID)
print(cc.progress_bar(p), "|", p["phase"], "|", cc.wait_hint(p))
print(f"failed {p['failed']}, elapsed {p['elapsed_h']} h, {p['slides_per_hour'] or '-'} slides/h, "
      f"driver {'running' if p['driver_running'] else 'not running'}, waiting in queue {p.get('queued', '-')}")
if isinstance(p.get("workers_now"), dict):
    display(pd.DataFrame([{"worker": w, "slide now": ", ".join(s) or "idle"} for w, s in sorted(p["workers_now"].items())]))
if p["by_worker"]:
    print("done per worker:", p["by_worker"])
if p["failures"]:
    print("failed slides:", p["failures"])
print("\n".join(["--- log"] + p["log_tail"]))

Optional: refresh automatically every few minutes until the run finishes (interrupt the kernel to stop watching; the run keeps going).

In [ ]:
WATCH = False
EVERY_MIN = 5
from IPython.display import clear_output
while WATCH:
    p = cc.progress(RUN_ID, live=False)
    clear_output(wait=True)
    print(time.strftime("%H:%M"), cc.progress_bar(p), "|", p["phase"], "|", cc.wait_hint(p), "| failed", p["failed"])
    if p["phase"] in ("finished", "error") or not p["driver_running"]:
        break
    time.sleep(EVERY_MIN * 60)

## 6. Results
After the phase reads `finished`: where everything is, and the run summary (label fields, feather counts, reviews).

In [ ]:
r = cc.results(RUN_ID)
print("labels/tables:", r["labels"]); print("crops/overlays:", r["segmentation"]); print("Drive:", r["drive"])
s = r["summary"]
if s:
    print(f"{s['slides']} slides, {s.get('feathers')} feathers, {s.get('duplicate_feathers')} duplicates, "
          f"{s.get('feathers_needing_review')} feathers to review, {s.get('possible_missed_feathers', 0)} possible missed, "
          f"mean feather score {s.get('feather_score_mean')}")
    display(pd.DataFrame(s["status_counts"]).fillna(0).astype(int).T)
    if s.get("gold"):
        display(pd.DataFrame({k: {c: v[c] for c in ("n", "correct", "accuracy", "wrong_value", "false_na")}
                              for k, v in s["gold"].items()}).T)
    print("Open label_lab.ipynb with RUN_DIR = '../data/label_runs/%s' and SEGMENTATION_DIR = '../data/sample_runs/%s' "
          "for the slide grid and review gallery." % (RUN_ID, RUN_ID))
else:
    print("no summary yet (the run has not finished)")

## 7. All runs started from here

In [ ]:
display(cc.runs())

## 8. Stop a run
Stops the run's driver here; slides already on a worker finish. `DROP_QUEUED = True` also removes the jobs still waiting (only do that if no other run is using the workers).

In [ ]:
STOP = False
DROP_QUEUED = False
if STOP:
    print(cc.stop_run(RUN_ID, DROP_QUEUED))